In [1]:
%pip install streamlit

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import os
import json
import requests
from dotenv import load_dotenv

load_dotenv()
HENRIK_KEY = os.getenv("HENRIK_API_KEY")
BASE = "https://api.henrikdev.xyz"
HEADERS = {"Authorization": HENRIK_KEY}

# --- Step 1: look up an account ---
NAME, TAG, REGION = "SPARROW", "020", "eu"      # replace with your own Riot ID

r = requests.get(f"{BASE}/valorant/v1/account/{NAME}/{TAG}", headers=HEADERS)
print("Account lookup:", r.status_code)
account = r.json()
print(json.dumps(account, indent=2)[:600])

In [ ]:
# --- Step 2: fetch recent matches ---
r = requests.get(
    f"{BASE}/valorant/v4/matches/{REGION}/pc/{NAME}/{TAG}",
    headers=HEADERS,
    params={"mode": "competitive", "size": 5}
)
print("Matches:", r.status_code)
matches = r.json()

# --- Step 3: inspect the structure of ONE player's stats ---
match = matches["data"][0]
print("Top-level keys in a match:", list(match.keys()))

players = match["players"]
print("\nPlayer entry keys:", list(players[0].keys()))
print("\nOne player's full record:")
print(json.dumps(players[0], indent=2)[:2000])

Matches: 200
Top-level keys in a match: ['metadata', 'players', 'observers', 'coaches', 'teams', 'rounds', 'kills']

Player entry keys: ['puuid', 'name', 'tag', 'team_id', 'platform', 'party_id', 'agent', 'stats', 'ability_casts', 'tier', 'customization', 'account_level', 'session_playtime_in_ms', 'behavior', 'economy']

One player's full record:
{
  "puuid": "a2bf6f6e-29bc-58df-8bf3-5e42cdd8e381",
  "name": "\u0623\u0646\u0633",
  "tag": "FTW",
  "team_id": "Red",
  "platform": "pc",
  "party_id": "44f12062-8cfd-49a8-af25-86fda7b0ed98",
  "agent": {
    "id": "a3bfb853-43b2-7238-a4f1-ad90e9e46bcc",
    "name": "Reyna"
  },
  "stats": {
    "score": 5471,
    "kills": 21,
    "deaths": 10,
    "assists": 4,
    "headshots": 22,
    "bodyshots": 30,
    "legshots": 0,
    "damage": {
      "dealt": 3875,
      "received": 2440
    }
  },
  "ability_casts": {
    "grenade": 24,
    "ability1": 8,
    "ability2": 9,
    "ultimate": 3
  },
  "tier": {
    "id": 15,
    "name": "Platinum 1"

In [ ]:
# Inspect the round and kill event structures
print("Rounds:", len(match["rounds"]))
print("Round keys:", list(match["rounds"][0].keys()))
print()

kills = match["kills"]
print("Total kill events:", len(kills))
print("Kill event keys:", list(kills[0].keys()))
print("\nOne kill event:")
print(json.dumps(kills[0], indent=2)[:1200])

Rounds: 19
Round keys: ['id', 'result', 'ceremony', 'winning_team', 'plant', 'defuse', 'stats']

Total kill events: 142
Kill event keys: ['time_in_round_in_ms', 'time_in_match_in_ms', 'round', 'killer', 'victim', 'assistants', 'location', 'weapon', 'secondary_fire_mode', 'player_locations']

One kill event:
{
  "time_in_round_in_ms": 5775,
  "time_in_match_in_ms": 60843,
  "round": 0,
  "killer": {
    "puuid": "14b37b87-9739-5c66-9760-728ccc16dd99",
    "name": "MrChaos",
    "tag": "123",
    "team": "Red"
  },
  "victim": {
    "puuid": "9fde26eb-dd07-5ed3-af40-ae6bf4a7474a",
    "name": "OTC",
    "tag": "4106",
    "team": "Blue"
  },
  "assistants": [],
  "location": {
    "x": 416,
    "y": -2767
  },
  "weapon": {
    "id": "1baa85b4-4c70-1284-64bb-6481dfc3bb4e",
    "name": "Ghost",
    "type": "Weapon"
  },
  "secondary_fire_mode": false,
  "player_locations": [
    {
      "player": {
        "puuid": "d84bedde-d43f-5b24-a6ab-e328280603d0",
        "name": "CTO",
        "ta

In [ ]:
from henrik_client import get_player_stats

result, error = get_player_stats(NAME, TAG, REGION, size=5)

if error:
    print("Error:", error)
else:
    print(result["meta"])
    print()
    for k, v in result["stats"].items():
        print(f"  {k:35s} {v}")

{'name': 'SPARROW#020', 'main_agent': 'waylay', 'matches_analysed': 5, 'rounds_analysed': 104}

  Average Combat Score                183.4
  Headshot %                          34.4
  Kills Per Round                     0.61
  Assists Per Round                   0.24
  Kills:Deaths                        0.74
  Average Damage Per Round            112.0
  First Kills Per Round               0.077
  First Deaths Per Round              0.115
  Kill, Assist, Trade, Survive %      58.7


In [ ]:
from analysis_engine import (
    load_clean_data, build_distributions,
    build_agent_distributions, compare_player
)

df = load_clean_data()
distributions = build_distributions(df)

player_stats = result["stats"]
main_agent = result["meta"]["main_agent"]

agent_dist = build_agent_distributions(df, main_agent)
print(f"Agent: {main_agent} — benchmark metrics available: {len(agent_dist)}")

comparison = compare_player(player_stats, distributions, agent_dist)
print(comparison.to_string(index=False))

Agent: waylay — benchmark metrics available: 11
                        metric  player  pro_median  percentile           verdict  agent_median  agent_percentile
          Average Combat Score 183.400      198.00        37.9      below median        202.00              30.4
                    Headshot %  34.400       28.00        76.5          strength         30.00              67.7
               Kills Per Round   0.610        0.69        31.6      below median          0.74              24.2
             Assists Per Round   0.240        0.24        47.3      below median          0.18              71.6
                  Kills:Deaths   0.740        0.99        21.4 priority weakness          1.00              18.1
      Average Damage Per Round 112.000      128.00        28.3      below median        130.00              23.5
         First Kills Per Round   0.077        0.09        40.9      below median          0.15              10.3
        First Deaths Per Round   0.115        0.

In [ ]:
from coaching_engine import comparison_to_text, generate_coaching

player_text = comparison_to_text(comparison, agent=main_agent.title())
coaching = generate_coaching(player_text)
print(coaching)

Your overall output currently sits below agent benchmarks, reflected in an Average Combat Score of 183.4 (30th percentile for Waylay players) and an Average Damage Per Round of 112.0 (24th percentile). 

Priority 1: Your Kill, Assist, Trade, Survive percentage is 58.7%, putting you in the 12th percentile for Waylay players against the agent median of 69.0%. Lower round participation makes it harder for your team to secure rounds consistently. 
Habit: Build the habit of calling out your swing to a teammate beforehand, ensuring every fight you take can be traded or assisted.

Priority 2: Your Kills:Deaths ratio sits at 0.74, placing you in the 18th percentile compared to the Waylay median of 1.0. Dying more often than securing kills reduces your team's numerical advantage. 
Drill: Spend 10 minutes in Deathmatch practicing slice-the-pie technique around corners to isolate 1v1 fights instead of wide-swinging into open space.

Your raw mechanics are performing well, with a 34.4% Headshot pe

In [ ]:
import importlib
import analysis_engine, coaching_engine
importlib.reload(analysis_engine)
importlib.reload(coaching_engine)

from analysis_engine import detect_patterns
from coaching_engine import comparison_to_text, generate_coaching

role, findings = detect_patterns(comparison, main_agent)
print("Role:", role)
for f in findings:
    print(" -", f)

player_text = comparison_to_text(comparison, agent=main_agent.title(),
                                 role=role, findings=findings)
coaching = generate_coaching(player_text)
print("\n" + coaching)

Role: duelist
 - Takes very few opening duels for a duelist while also dying first less often than typical for the agent. This indicates passive play in a role that requires creating space, rather than a discipline problem. Advice to play more cautiously would be counterproductive.
 - Aim is above typical for the agent while overall trade efficiency is well below it, suggesting the issue lies in positioning and timing rather than mechanical skill.

Your mechanical aim is sharp, but your overall round impact is currently limited, placing your 0.74 K/D ratio in the 18th percentile among Waylay players.

Priority 1: Opening duel involvement. Your First Kills Per Round of 0.077 sits in the bottom 10% of Waylay players, showing you are playing too passively for a duelist tasked with creating space. Build a habit of committing to an early entry duel in the first 15 seconds of the round, calling for a teammate's flash or utility to help you initiate the fight.

Priority 2: Positioning and tra

In [ ]:
import importlib
import analysis_engine, coaching_engine, experiment_log
importlib.reload(analysis_engine); importlib.reload(coaching_engine); importlib.reload(experiment_log)

from analysis_engine import (load_clean_data, build_distributions,
                             build_agent_distributions, compare_player, detect_patterns)
from coaching_engine import comparison_to_text, generate_coaching
from henrik_client import get_player_stats

NAME, TAG, REGION = "Sparrow", "020", "eu"      # change if needed

result, error = get_player_stats(NAME, TAG, REGION, size=10)
if error:
    raise RuntimeError(error)

df = load_clean_data()
distributions = build_distributions(df)

stats = result["stats"]
agent = result["meta"]["main_agent"]
agent_dist = build_agent_distributions(df, agent)
comparison = compare_player(stats, distributions, agent_dist)
role, findings = detect_patterns(comparison, agent)

print("Agent:", agent, "| Role:", role)
print("Findings detected:", len(findings))
for f in findings:
    print(" -", f)
print()
print(comparison.to_string(index=False))

Agent: jett | Role: duelist
Findings detected: 1
 - Takes very few opening duels for a duelist while also dying first less often than typical for the agent. This indicates passive play in a role that requires creating space, rather than a discipline problem. Advice to play more cautiously would be counterproductive.

                        metric  player  pro_median  percentile           verdict  agent_median  agent_percentile
          Average Combat Score 175.300      198.00        31.6      below median        225.00              14.1
                    Headshot %  26.000       28.00        36.6      below median         29.00              33.0
               Kills Per Round   0.600        0.69        29.7      below median          0.80              11.5
             Assists Per Round   0.250        0.24        50.3      above median          0.14              92.6
                  Kills:Deaths   0.770        0.99        24.8 priority weakness          1.08              12.0
   

In [4]:
text_without = comparison_to_text(comparison, agent=agent.title())   # no role, no findings

print("=== INPUT CHECK ===")
print(text_without[:300])
print("\n=== OUTPUT ===")
out_without = generate_coaching(text_without)
print(out_without)

=== INPUT CHECK ===
Main agent: Jett

Metric | Player | Pro median | Percentile | Agent median | Agent percentile | Verdict
Average Combat Score | 175.3 | 198.0 | 31.6th | 225.0 | 14.1 | below median
Headshot % | 26.0 | 28.0 | 36.6th | 29.0 | 33.0 | below median
Kills Per Round | 0.6 | 0.69 | 29.7th | 0.8 | 11.5 | belo

=== OUTPUT ===
Your recent performance on Jett shows an Average Combat Score of 175.3 (14th Jett percentile) and an Average Damage Per Round of 115.0 (14th Jett percentile). As a Duelist, your current damage output and overall combat impact are sitting below agent benchmarks.

**Priority 1:** Kill:Death Ratio. Your K:D sits at 0.77, placing you in the 12th percentile of Jett players. Winning duels is essential for maintaining site control and team momentum. 
*Drill:* Run two Deathmatches daily strictly focused on pre-aiming common sightlines and firing 2-bullet rifle bursts, using Jett's dash immediately after taking an opening duel to break line of sight.

**Priority 2

In [5]:
text_with = comparison_to_text(comparison, agent=agent.title(), role=role, findings=findings)

print("=== INPUT CHECK ===")
print(text_with[:600])
print("\n=== OUTPUT ===")
out_with = generate_coaching(text_with)
print(out_with)

=== INPUT CHECK ===
Main agent: Jett (role: duelist)

Metric | Player | Pro median | Percentile | Agent median | Agent percentile | Verdict
Average Combat Score | 175.3 | 198.0 | 31.6th | 225.0 | 14.1 | below median
Headshot % | 26.0 | 28.0 | 36.6th | 29.0 | 33.0 | below median
Kills Per Round | 0.6 | 0.69 | 29.7th | 0.8 | 11.5 | below median
Assists Per Round | 0.25 | 0.24 | 50.3th | 0.14 | 92.6 | above median
Kills:Deaths | 0.77 | 0.99 | 24.8th | 1.08 | 12.0 | priority weakness
Average Damage Per Round | 115.0 | 128.0 | 31.9th | 145.0 | 13.5 | below median
First Kills Per Round | 0.066 | 0.09 | 36.2th | 0.19 | 

=== OUTPUT ===
Your output on Jett sits below the agent median, with an Average Combat Score of 175.3 and Average Damage Per Round of 115.0 placing you in the 14th percentile of Jett players. The data indicates a passive approach on an agent designed to aggressively enter sites and create space.

Priority 1: Opening duels and space creation. Your First Kills Per Round of 0.06

In [7]:
"""Experiment logging for prompt development."""

import json
import os
from datetime import datetime

import pandas as pd

LOG_DIR = "prompt_experiments"
LOG_PATH = f"{LOG_DIR}/log.json"


def save_experiment(version, prompt_text, player_text, output, model, notes=""):
    """Save a prompt version and its output, as JSON and as readable text."""
    os.makedirs(LOG_DIR, exist_ok=True)

    log = []
    if os.path.exists(LOG_PATH):
        with open(LOG_PATH) as f:
            log = json.load(f)

    entry = {
        "version": version,
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "model": model,
        "prompt": prompt_text,
        "input_data": player_text,
        "output": output,
        "notes": notes,
        "word_count": len(output.split()),
    }

    log = [e for e in log if e["version"] != version]
    log.append(entry)
    log.sort(key=lambda e: e["version"])

    with open(LOG_PATH, "w") as f:
        json.dump(log, f, indent=2)

    with open(f"{LOG_DIR}/{version}_output.txt", "w", encoding="utf-8") as f:
        f.write(f"VERSION: {version}\nMODEL: {model}\nTIME: {entry['timestamp']}\n")
        f.write(f"WORD COUNT: {entry['word_count']}\n\n")
        f.write(f"--- PROMPT ---\n{prompt_text}\n\n")
        f.write(f"--- INPUT DATA ---\n{player_text}\n\n")
        f.write(f"--- OUTPUT ---\n{output}\n")

    print(f"Saved {version} ({entry['word_count']} words)")
    return entry


def load_log():
    if not os.path.exists(LOG_PATH):
        return []
    with open(LOG_PATH) as f:
        return json.load(f)


def experiment_table():
    log = load_log()
    if not log:
        return pd.DataFrame()
    return pd.DataFrame([{
        "version": e["version"],
        "timestamp": e["timestamp"][:16],
        "model": e["model"],
        "words": e["word_count"],
        "notes": e["notes"],
    } for e in log])

In [8]:
save_experiment(version="v2_live", prompt_text=coaching_engine.SYSTEM_PROMPT,
                player_text=text_without, output=out_without, model="gemini-3.6-flash",
                notes="WRITE WHAT YOU OBSERVED")

save_experiment(version="v3_live", prompt_text=coaching_engine.SYSTEM_PROMPT,
                player_text=text_with, output=out_with, model="gemini-3.6-flash",
                notes="WRITE WHAT YOU OBSERVED")

experiment_table()

Saved v2_live (193 words)
Saved v3_live (208 words)


,version,timestamp,model,words,notes
0,v1,2026-09-04T00:34,gemini-3.6-flash,171,"Baseline prompt, temperature 0.3. Correctly tr..."
1,v2,2026-09-04T00:35,gemini-3.6-flash,171,Added three rules to the baseline prompt: perc...
2,v2_live,2026-09-28T15:22,gemini-3.6-flash,193,WRITE WHAT YOU OBSERVED
3,v3,2026-09-04T00:35,gemini-3.6-flash,171,Added role-based pattern detection in analysis...
4,v3_live,2026-09-28T15:22,gemini-3.6-flash,208,WRITE WHAT YOU OBSERVED
